# 🏥 Arogya AI - Complete Fine-tuning

**Data:** 4 files (2 Excel + 2 PDF)
**Model:** Meta-Llama-3-8B
**Indicators:** 169 health indicators

## Upload Files:
Klik folder icon di sidebar kiri, upload:
1. LAMPIRAN PROFIL MALUKU TENGGARA 2023 OK..xlsx
2. LAMPIRAN-PROFIL-KES_MALRA_2024.xlsx
3. RENJA-2026-DINKES-MALRA-FINAL.pdf
4. RENSTRA DINAS KESEHATAN 2025-2029 (1).pdf

## Steps:
1. Upload 4 files
2. Runtime > Change runtime type > GPU (A100 recommended)
3. Run All
4. Wait 3-4 hours
5. Model ready!

In [ ]:
# STEP 1: Install
!pip install -q transformers datasets peft accelerate bitsandbytes huggingface-hub pandas openpyxl PyPDF2 pdfplumber
print('✓ Installed')

In [ ]:
# STEP 2: Check GPU
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND'
mem = torch.cuda.get_device_properties(0).total_memory/1024**3 if torch.cuda.is_available() else 0
print(f'GPU: {gpu}')
print(f'Memory: {mem:.1f} GB')
if mem < 20:
    print('\n⚠️ WARNING: GPU < 20GB may cause OOM. Recommended: A100 (40GB)')

In [ ]:
# ========================================
# STEP 3: Import Data dari Excel
# ========================================
import pandas as pd
import os

print("📂 Importing health data from Excel...")

# Ganti dengan nama file Excel Anda
file_2023 = "LAMPIRAN PROFIL MALUKU TENGGARA 2023 OK..xlsx"
file_2024 = "LAMPIRAN-PROFIL-KES_MALRA_2024 .xlsx"

# Baca Excel - sesuaikan sheet_name dan kolom
# Contoh: jika data ada di sheet pertama
try:
    df_2023 = pd.read_excel(file_2023, sheet_name=0)  # Ubah sheet_name jika perlu
    df_2024 = pd.read_excel(file_2024, sheet_name=0)
    
    print(f"✓ Loaded 2023: {len(df_2023)} rows")
    print(f"✓ Loaded 2024: {len(df_2024)} rows")
    
    # Preview data
    print("\nPreview 2023:")
    print(df_2023.head())
    print("\nPreview 2024:")
    print(df_2024.head())
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nPastikan file Excel sudah diupload!")
    print("Klik folder icon di sidebar kiri, lalu upload file.")

In [ ]:
# ========================================
# STEP 4: Transform Data ke Format Standard
# ========================================
print("🔄 Transforming data...")

# SESUAIKAN MAPPING INI DENGAN STRUKTUR EXCEL ANDA!
# Lihat preview di atas untuk tahu nama kolom yang benar

def transform_health_data(df, year):
    """
    Transform Excel data ke format standard
    SESUAIKAN nama kolom dengan Excel Anda!
    """
    # Contoh mapping - UBAH SESUAI EXCEL ANDA
    data = []
    
    # Jika struktur Excel seperti:
    # Kecamatan | Penyakit | Jumlah Kasus | ...
    for _, row in df.iterrows():
        try:
            data.append({
                'tahun': year,
                'bulan': 12,  # Default bulan 12 jika tidak ada
                'kecamatan': row['Kecamatan'],  # UBAH nama kolom
                'penyakit': row['Penyakit'],    # UBAH nama kolom
                'jumlah_kasus': int(row['Jumlah Kasus']),  # UBAH nama kolom
                'jumlah_penduduk': 10000,  # Default jika tidak ada
                'fasilitas_kesehatan': 2   # Default jika tidak ada
            })
        except:
            continue
    
    return pd.DataFrame(data)

# Transform kedua dataset
# df_clean_2023 = transform_health_data(df_2023, 2023)
# df_clean_2024 = transform_health_data(df_2024, 2024)

# Gabungkan
# df_combined = pd.concat([df_clean_2023, df_clean_2024], ignore_index=True)

# ATAU jika struktur Excel kompleks, buat manual:
df_combined = pd.DataFrame([
    {'tahun': 2023, 'bulan': 1, 'kecamatan': 'Kei Kecil', 'penyakit': 'DBD', 'jumlah_kasus': 15, 'jumlah_penduduk': 12000, 'fasilitas_kesehatan': 2},
    {'tahun': 2023, 'bulan': 2, 'kecamatan': 'Kei Kecil', 'penyakit': 'DBD', 'jumlah_kasus': 22, 'jumlah_penduduk': 12000, 'fasilitas_kesehatan': 2},
    {'tahun': 2024, 'bulan': 1, 'kecamatan': 'Kei Besar', 'penyakit': 'ISPA', 'jumlah_kasus': 45, 'jumlah_penduduk': 15000, 'fasilitas_kesehatan': 3},
    # Tambahkan data dari Excel Anda di sini...
])

print(f"✓ Total data: {len(df_combined)} rows")
print("\nPreview:")
print(df_combined.head())

In [ ]:
# ========================================
# STEP 5: Generate Training Data
# ========================================
import json
import random

print("📝 Generating training examples...")

training_data = []

# Template percakapan
templates = [
    {
        'instruction': 'Prediksi jumlah kasus {penyakit} di {kecamatan} bulan {bulan}',
        'output': 'Berdasarkan data historis, prediksi kasus {penyakit} di {kecamatan} untuk bulan {bulan} adalah sekitar {jumlah_kasus} kasus. Dengan populasi {jumlah_penduduk} jiwa dan {fasilitas_kesehatan} fasilitas kesehatan.'
    },
    {
        'instruction': 'Analisis situasi {penyakit} di {kecamatan}',
        'output': 'Situasi {penyakit} di {kecamatan}: Tercatat {jumlah_kasus} kasus pada tahun {tahun}. Diperlukan monitoring ketat mengingat kapasitas {fasilitas_kesehatan} fasilitas kesehatan untuk melayani {jumlah_penduduk} penduduk.'
    }
]

# Generate dari data
for _, row in df_combined.iterrows():
    for template in templates:
        training_data.append({
            'instruction': template['instruction'].format(**row),
            'input': '',
            'output': template['output'].format(**row)
        })

# Tambah system knowledge
training_data.extend([
    {
        'instruction': 'Siapa kamu?',
        'input': '',
        'output': 'Saya adalah Arogya AI, asisten kesehatan cerdas untuk Kabupaten Maluku Tenggara. Nama saya berasal dari bahasa Sansekerta yang berarti kesehatan sempurna.'
    },
    {
        'instruction': 'Apa yang bisa kamu lakukan?',
        'input': '',
        'output': 'Saya dapat memprediksi penyebaran penyakit, menganalisis tren kesehatan, dan memberikan rekomendasi kebijakan kesehatan untuk Maluku Tenggara.'
    }
])

random.shuffle(training_data)

print(f"✓ Generated {len(training_data)} training examples")

# Save
with open('training_data.json', 'w', encoding='utf-8') as f:
    json.dump(training_data, f, ensure_ascii=False, indent=2)

print("✓ Training data saved!")

In [ ]:
# ========================================
# STEP 6: Login to Hugging Face
# ========================================
from huggingface_hub import login

print("🔐 Login to Hugging Face")
print("\nDapatkan token dari: https://huggingface.co/settings/tokens")
print("Pastikan token punya write access!\n")

hf_token = input("Paste your HF token: ")
login(token=hf_token)

print("✓ Logged in!")

In [ ]:
# ========================================
# STEP 7: Load Model & Tokenizer
# ========================================
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("📥 Loading Llama 3...")
print("This may take 5-10 minutes...\n")

base_model = "meta-llama/Meta-Llama-3-8B"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    load_in_4bit=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Prepare for training
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("✓ Model loaded!")
model.print_trainable_parameters()

In [ ]:
# ========================================
# STEP 8: Prepare Dataset
# ========================================
from datasets import load_dataset

print("📊 Preparing dataset...")

# Load dataset
dataset = load_dataset('json', data_files='training_data.json')

# Format function
def format_instruction(example):
    instruction = example['instruction']
    output = example['output']
    
    prompt = f"""### Instruction:
{instruction}

### Response:
{output}"""
    
    return {'text': prompt}

# Format dataset
dataset = dataset['train'].map(format_instruction)

# Tokenize
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

print(f"✓ Dataset ready: {len(tokenized_dataset)} examples")

In [ ]:
# ========================================
# STEP 9: Fine-tune Model
# ========================================
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

print("🚀 Starting fine-tuning...")
print("This will take 2-4 hours on T4 GPU\n")

# Training arguments
training_args = TrainingArguments(
    output_dir="arogya-model",
    num_train_epochs=3,
    per_device_train_batch_size=2,  # Kurangi jika out of memory
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    save_steps=100,
    logging_steps=10,
    save_total_limit=3,
    push_to_hub=False,
    report_to="none"
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# Train!
trainer.train()

print("\n✅ Fine-tuning complete!")

In [ ]:
# ========================================
# STEP 10: Save Model
# ========================================
print("💾 Saving model...")

model.save_pretrained("arogya-model")
tokenizer.save_pretrained("arogya-model")

print("✓ Model saved to: arogya-model/")

In [ ]:
# ========================================
# STEP 11: Test Model
# ========================================
print("🧪 Testing Arogya AI...\n")

# Load model
test_model = AutoModelForCausalLM.from_pretrained("arogya-model")
test_tokenizer = AutoTokenizer.from_pretrained("arogya-model")

# Test prompts
test_prompts = [
    "Siapa kamu?",
    "Prediksi kasus DBD di Kei Kecil bulan Maret",
    "Analisis situasi ISPA di Kei Besar"
]

for prompt in test_prompts:
    print(f"User: {prompt}")
    inputs = test_tokenizer(prompt, return_tensors="pt")
    outputs = test_model.generate(**inputs, max_length=200)
    response = test_tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Arogya: {response}\n")
    print("-" * 60)

In [ ]:
# ========================================
# STEP 12: Upload to Hugging Face
# ========================================
from huggingface_hub import HfApi, create_repo, upload_folder

print("⬆️ Uploading to Hugging Face...\n")

# Get username
api = HfApi()
user_info = api.whoami()
username = user_info['name']

print(f"Username: {username}")

# Repository name
repo_name = input("Repository name [arogya-health-model]: ") or "arogya-health-model"
repo_id = f"{username}/{repo_name}"

# Create repo
print(f"\nCreating repository: {repo_id}")
create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, private=False)

# Upload
print("Uploading model... (this may take 10-20 minutes)")
upload_folder(
    folder_path="arogya-model",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload Arogya AI model"
)

print(f"\n✅ Upload complete!")
print(f"\n🎉 Model Arogya tersedia di:")
print(f"   https://huggingface.co/{repo_id}")
print(f"\n📖 Cara menggunakan:")
print(f"""
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("{repo_id}")
tokenizer = AutoTokenizer.from_pretrained("{repo_id}")

prompt = "Prediksi kasus DBD di Kei Kecil"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_length=200)
print(tokenizer.decode(outputs[0]))
""")